# Tuna-TTS -- Kaggle setup + smoke test

Run this notebook FIRST, top to bottom, before the training notebook. It:
1. Verifies the GPU(s) and installs dependencies.
2. Downloads the base model, Khmer tokenizer, and dataset from Hugging Face.
3. Runs the PLAN.md section 21.5 codec spot-check and a real
   forward+backward smoke test on the actual model/data/checkpoint path.

**Before running:** turn **Internet** ON in notebook settings (needed to
clone the repo and download from Hugging Face). A single T4 GPU is enough
for this notebook (the 2-GPU run is in the training notebook).

If every cell below prints PASS, go run `Tuna_TTS_Kaggle_Train.ipynb`.


## 0. Config

In [ ]:
# EDIT THIS if you fork/rename the repo.
GITHUB_REPO_URL = "https://github.com/Pich09/Tuna-TTS.git"

REPO_DIR = "/kaggle/working/Tuna-tts"


## 1. Environment check

Confirms the accelerator is on and (for the training notebook) that both
GPUs are visible. If this prints 0 GPUs, go to
**Notebook settings (right sidebar) -> Accelerator -> GPU T4 x2** and
re-run.


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


## 2. Clone the project code

Cloned straight from GitHub into `/kaggle/working/Tuna-tts` (a writable
directory -- training writes checkpoints/logs next to the code). No
dataset upload needed for code changes; just push to GitHub and re-run
this cell to pick them up.

If the repo is private, add a Kaggle Secret named `GITHUB_TOKEN` (a
fine-grained PAT with read-only access to just this repo) -- the cell
below picks it up automatically and never prints or hardcodes it. Public
repo: leave the secret unset, nothing else to do.


In [ ]:
import shutil, subprocess, os

try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    github_token = None  # fine for a public repo

clone_url = GITHUB_REPO_URL
if github_token:
    clone_url = GITHUB_REPO_URL.replace("https://", f"https://{github_token}@")

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", clone_url, REPO_DIR], check=True)
print(f"Cloned {GITHUB_REPO_URL} -> {REPO_DIR}")


## 3. Install dependencies

Kaggle ships a recent torch+CUDA already (kept as-is -- reinstalling torch
here would be slow and is unnecessary). Everything else installs from
`requirements.txt`, which pins fish-speech to the exact commit
(`d3df505`) this codebase's tokenizer/model code was verified against --
main HEAD is NOT compatible with openaudio-s1-mini (see requirements.txt's
comments). Requires this notebook's **Internet** toggle to be ON.


In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
# protobuf resolver conflict (descript-audiotools wants <3.20, fish-speech's
# generated _pb2.py needs >=3.20) -- upgrade separately after the main install,
# same fix used in local development (see requirements.txt).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "protobuf>=3.20,<6"], check=True)
print("Dependencies installed.")


## 4. Download the base model, Khmer tokenizer, and training data

Downloaded here (on Kaggle's fast network) rather than uploaded from a slow
home connection. Add an `HF_TOKEN` secret via the notebook's **Add-ons ->
Secrets** menu -- needed if `fishaudio/openaudio-s1-mini` or
`Panhapich/khmer-tts-processed` are gated/private, AND (training notebook
only) for uploading checkpoints to `Panhapich/Tuna-TTS` -- that token needs
**write** access to that repo specifically. Never paste a token directly
into a cell; this reads it from the secret and also exports it as the
`HF_TOKEN` environment variable so the training subprocess launched later
inherits it the same way.


In [ ]:
import os
from huggingface_hub import snapshot_download, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None  # fine if the repos are public and no secret is set

if hf_token:
    os.environ["HF_TOKEN"] = hf_token  # inherited by the torchrun subprocess later

ckpt_dir = f"{REPO_DIR}/checkpoints/openaudio-s1-mini"
snapshot_download(repo_id="fishaudio/openaudio-s1-mini", local_dir=ckpt_dir, token=hf_token)
print(f"Base model -> {ckpt_dir}")

hf_hub_download(repo_id="Panhapich/khmer-sp-8k", filename="khmer-sp-8k.model",
                 local_dir=f"{REPO_DIR}/data", token=hf_token)
print("Khmer tokenizer -> data/khmer-sp-8k.model")

snapshot_download(repo_id="Panhapich/khmer-tts-processed", repo_type="dataset",
                   local_dir=f"{REPO_DIR}/data/protos", token=hf_token)
print(f"Training data -> {REPO_DIR}/data/protos")

# fish_speech.models.dac.inference does pyrootutils.setup_root(indicator=".project-root")
# at import time, which fails when fish-speech is pip-installed rather than a
# repo checkout -- same fix used in local development.
import fish_speech.models.dac as dac_pkg
marker_path = os.path.join(os.path.dirname(dac_pkg.__file__), ".project-root")
open(marker_path, "a").close()
print(f"Touched {marker_path}")


## 5. Locate the actual train/validation .protos subdirectories

`Panhapich/khmer-tts-processed`'s exact folder layout inside the downloaded
snapshot can vary; this finds the train/validation dirs so the config
doesn't need hand-editing.


In [ ]:
import glob, os

protos_root = f"{REPO_DIR}/data/protos"
train_dir = next((d for d in glob.glob(f"{protos_root}/**/train", recursive=True) if os.path.isdir(d)), None) \
    or next((d for d in glob.glob(f"{protos_root}/**/*train*", recursive=True) if os.path.isdir(d)), protos_root)
val_dir = next((d for d in glob.glob(f"{protos_root}/**/val*", recursive=True) if os.path.isdir(d)), protos_root)

print(f"proto_train_dir -> {train_dir}")
print(f"proto_val_dir   -> {val_dir}")

# Patch the experiment config in place so scripts/train.py picks these paths
# up as-is (avoids hand-editing YAML for a path that may differ per run).
import re
for cfg_name in ["exp001.yaml", "exp001_kaggle.yaml"]:
    cfg_path = f"{REPO_DIR}/configs/experiments/{cfg_name}"
    text = open(cfg_path).read()
    text = re.sub(r"proto_train_dir: .*", f"proto_train_dir: {os.path.relpath(train_dir, REPO_DIR)}", text)
    text = re.sub(r"proto_val_dir: .*", f"proto_val_dir: {os.path.relpath(val_dir, REPO_DIR)}", text)
    open(cfg_path, "w").write(text)
print("Patched proto_train_dir / proto_val_dir in exp001*.yaml")


## 6. Codec spot-check (PLAN.md section 21.5)

Decodes a handful of the dataset's precomputed semantic codes straight
through the S1-mini codec, with no model involved -- confirms the
downloaded checkpoint/data are actually compatible with each other before
spending any GPU time on training.


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/codec_spotcheck.py",
     "--proto-dir", "data/protos",
     "--codec-checkpoint", "checkpoints/openaudio-s1-mini/codec.pth",
     "--num-samples", "5",
     "--out-dir", "/kaggle/working/codec_spotcheck"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Codec spot-check FAILED -- do not proceed to training, see output above."
print("[PASS] codec spot-check")


## 7. Forward/backward smoke test (PLAN.md section 43)

Builds the real model (S1-mini + Khmer embedding extension + LoRA), runs a
handful of real training batches through forward + backward + optimizer
step on a single GPU, and checks the loss and every gradient are finite.
This is the same check as `scripts/test_model_protos.py`, run inline here
so its assertion stops the notebook before touching the expensive
multi-GPU training cell below.


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/test_model_protos.py",
     "--base-model-path", "checkpoints/openaudio-s1-mini",
     "--khmer-sp-model", "data/khmer-sp-8k.model",
     "--proto-train-dir", "data/protos/train",
     "--num-batches", "5", "--batch-size", "1"],
    cwd=REPO_DIR, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Forward/backward smoke test FAILED -- see output above. Do not proceed to training."
print("\n[PASS] forward/backward smoke test -- safe to proceed to the training notebook.")


## All checks passed. Proceed to `Tuna_TTS_Kaggle_Train.ipynb`.